# 逻辑推理 完整教程：从直觉到严密推理

## 📚 教学目标
1. 掌握**系统化逻辑推理**的方法：约束分析 → 状态空间 → 最优解
2. 学会使用**决策树**分析信息获取效率（天平称重问题）
3. 理解**状态空间搜索**在约束满足问题中的应用（过河问题）
4. 通过 Python 实现 BFS/穷举来**验证**推理结论的正确性

**参考书**：《A Practical Guide to Quantitative Finance Interviews》(Xinfeng Zhou) 第2章

**教学策略**：每个谜题先建立约束条件，再通过系统推理找到最优解，最后用 Python 验证

---

## 1. 逻辑推理的核心框架

### 🎯 什么是逻辑推理题？

逻辑推理题不需要高深的数学公式，而是考察你能否：

```
1. 准确理解约束条件
2. 系统地枚举所有可能性
3. 用逻辑排除不可能的选项
4. 找到最优策略（最少步骤/次数）
```

### 💡 通用解题框架

| 步骤 | 内容 | 工具 |
|------|------|------|
| 1. 建模 | 将问题转化为数学模型 | 状态空间、决策树、图 |
| 2. 约束 | 明确所有限制条件 | 列表、不等式 |
| 3. 搜索 | 系统地探索解空间 | BFS、DFS、穷举 |
| 4. 优化 | 在所有可行解中找最优 | 比较、剪枝 |
| 5. 验证 | 用 Python 确认结论 | 模拟、暴力搜索 |

本教程通过三个经典谜题演示：

| 谜题 | 核心技巧 | 信息论视角 |
|------|----------|------------|
| 12球称重 | 决策树 + 三分法 | 每次称重获取 $\log_2 3$ 比特信息 |
| 25匹马赛跑 | 锦标赛排除法 | 每场比赛排除固定数量的候选 |
| 过河问题 | 状态空间 BFS | 图搜索最短路径 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations, product
from collections import deque

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 谜题一：12 球称重 (Defective Ball)

### 🎯 场景设定

你有 **12 个外观相同的球**，其中有且仅有 **1 个次品球**，它的重量与其他球不同（可能偏重或偏轻）。

你有一个**天平**（没有砝码，只能比较两组球的重量），可以使用 **3 次**。

**问题**：找出次品球，并判断它是偏重还是偏轻。

### 📐 信息论分析

- 可能的情况数：12 个球 × 2 种可能（重/轻）= **24 种**
- 每次称重有 3 种结果：左重、平衡、右重
- 3 次称重最多区分：$3^3 = 27$ 种情况

$$24 \leq 27 = 3^3$$

所以理论上 3 次**刚好够**！但必须精心设计每一步。

### 💡 关键约束

```
- 不知道次品是重还是轻
- 每次称重把球分成 3 组: 左盘、右盘、不称
- 理想策略: 每次将可能性数量缩小为原来的 1/3
```

In [ ]:
# ========== 步骤 1: 建立状态表示 ==========
print("📊 步骤 1: 分析可能的状态数")
print("=" * 50)
print(f"  球的数量: 12")
print(f"  次品可能性: 偏重 或 偏轻")
print(f"  总可能情况: 12 x 2 = 24 种")
print(f"")
print(f"  称重次数: 3 次")
print(f"  每次称重结果: 3 种 (左重/平衡/右重)")
print(f"  总可区分情况: 3^3 = {3**3} 种")
print(f"")
print(f"  24 <= 27 → 理论上 3 次足够!")
print(f"")
print(f"  💡 信息论视角:")
print(f"    需要信息量: log2(24) = {np.log2(24):.2f} 比特")
print(f"    每次称重提供: log2(3) = {np.log2(3):.2f} 比特")
print(f"    3次总信息量: 3 x log2(3) = {3 * np.log2(3):.2f} 比特")
print(f"    {3 * np.log2(3):.2f} >= {np.log2(24):.2f} → 信息够用")

In [ ]:
# ========== 步骤 2: 第一次称重策略 ==========
print("📊 步骤 2: 第一次称重 - 4 vs 4")
print("=" * 50)
print("  将 12 个球分为 3 组:")
print("    左盘: 球 1, 2, 3, 4")
print("    右盘: 球 5, 6, 7, 8")
print("    不称: 球 9, 10, 11, 12")
print()
print("  三种结果:")
print("  ┌─────────────────────────────────────────────────┐")
print("  │ 结果A: 左 = 右 (平衡)                           │")
print("  │   → 球1-8 都是正常球                            │")
print("  │   → 次品在球 9,10,11,12 中                      │")
print("  │   → 剩余可能: 4x2 = 8 种                        │")
print("  ├─────────────────────────────────────────────────┤")
print("  │ 结果B: 左 > 右 (左重)                           │")
print("  │   → 次品要么在左盘(偏重) 或 右盘(偏轻)          │")
print("  │   → 球9-12 确定正常                             │")
print("  │   → 剩余可能: 4+4 = 8 种                        │")
print("  ├─────────────────────────────────────────────────┤")
print("  │ 结果C: 左 < 右 (右重)                           │")
print("  │   → 与结果B对称                                  │")
print("  │   → 剩余可能: 4+4 = 8 种                        │")
print("  └─────────────────────────────────────────────────┘")
print()
print("  💡 每种结果剩 8 种可能 (总 24 缩减为 8 = 24/3)")
print("     完美的三分法!")

In [ ]:
# ========== 步骤 3: 详解情况A (第一次平衡) ==========
print("📊 步骤 3: 情况A详解 - 第一次平衡, 次品在 9-12 中")
print("=" * 60)
print("  已知: 球 1-8 正常, 次品在 {9, 10, 11, 12}")
print("  可能: 9重/9轻/10重/10轻/11重/11轻/12重/12轻 (8种)")
print()
print("  第二次称重: 球9, 10, 11 vs 球1, 2, 3 (已知正常球)")
print("  ┌─────────────────────────────────────────────────────┐")
print("  │ 结果A1: 平衡                                        │")
print("  │   → 9,10,11 都正常, 次品是球 12                     │")
print("  │   → 第三次: 球12 vs 球1 (正常球)                    │")
print("  │     重 → 12偏重; 轻 → 12偏轻                        │")
print("  ├─────────────────────────────────────────────────────┤")
print("  │ 结果A2: 左重 (9,10,11 重于 1,2,3)                   │")
print("  │   → 次品在 {9,10,11} 中且偏重 (3种可能)             │")
print("  │   → 第三次: 球9 vs 球10                             │")
print("  │     左重 → 9偏重; 右重 → 10偏重; 平衡 → 11偏重      │")
print("  ├─────────────────────────────────────────────────────┤")
print("  │ 结果A3: 左轻 (9,10,11 轻于 1,2,3)                   │")
print("  │   → 次品在 {9,10,11} 中且偏轻 (3种可能)             │")
print("  │   → 第三次: 球9 vs 球10                             │")
print("  │     左轻 → 9偏轻; 右轻 → 10偏轻; 平衡 → 11偏轻      │")
print("  └─────────────────────────────────────────────────────┘")
print()
print("  💡 每个分支最多 3 种可能, 1 次称重刚好区分!")

In [ ]:
# ========== 步骤 4: 详解情况B (第一次左重) ==========
print("📊 步骤 4: 情况B详解 - 第一次左重 (1,2,3,4 > 5,6,7,8)")
print("=" * 60)
print("  已知: 球9-12正常")
print("  可能: 1重/2重/3重/4重/5轻/6轻/7轻/8轻 (8种)")
print()
print("  第二次称重: 球1, 2, 5 vs 球3, 6, 9")
print("  (关键: 交换一些球 + 加入已知正常球)")
print("  ┌─────────────────────────────────────────────────────────┐")
print("  │ 结果B1: 平衡                                            │")
print("  │   → 1,2,3,5,6 都正常                                   │")
print("  │   → 次品在 {4(重), 7(轻), 8(轻)} 中 (3种)              │")
print("  │   → 第三次: 球7 vs 球8                                  │")
print("  │     左轻 → 7偏轻; 右轻 → 8偏轻; 平衡 → 4偏重           │")
print("  ├─────────────────────────────────────────────────────────┤")
print("  │ 结果B2: 左重 (1,2,5 > 3,6,9)                           │")
print("  │   球5从右→左, 天平仍左重 → 球5可排除(如果5轻,移到左      │")
print("  │     边应使左变轻, 但左仍重 → 5不是轻的次品)             │")
print("  │   球3从左→右, 天平仍左重 → 如果3重,移走应使左变轻 → 3不重│")
print("  │   → 次品在 {1(重), 2(重), 6(轻)} 中 (3种)              │")
print("  │   → 第三次: 球1 vs 球2                                  │")
print("  │     左重 → 1偏重; 右重 → 2偏重; 平衡 → 6偏轻           │")
print("  ├─────────────────────────────────────────────────────────┤")
print("  │ 结果B3: 左轻 (1,2,5 < 3,6,9)                           │")
print("  │   天平翻转了! 分析类似                                   │")
print("  │   → 次品在 {3(重), 5(轻)} 中... 但只有2种 → 也OK        │")
print("  │   → 第三次: 球5 vs 球9(正常)                            │")
print("  │     左轻 → 5偏轻; 平衡 → 3偏重                          │")
print("  └─────────────────────────────────────────────────────────┘")
print()
print("  💡 通过'交换球+加入正常球', 每种结果都缩减到 ≤3 种可能")

In [ ]:
# ========== 步骤 5: Python 实现完整的称重策略 ==========
def solve_12_balls(defective_ball, is_heavy):
    """
    模拟3次称重找出次品球
    
    参数:
        defective_ball: 次品球编号 (1-12)
        is_heavy: True=偏重, False=偏轻
    返回:
        (found_ball, found_heavy, weighings): 找到的球, 重/轻, 称重过程
    """
    # 每个球的重量: 正常=10, 偏重=11, 偏轻=9
    weights = {i: 10 for i in range(1, 13)}
    weights[defective_ball] = 11 if is_heavy else 9
    
    def weigh(left, right):
        """称重: 返回 'L'(左重), 'R'(右重), 'B'(平衡)"""
        l = sum(weights[b] for b in left)
        r = sum(weights[b] for b in right)
        if l > r: return 'L'
        elif l < r: return 'R'
        else: return 'B'
    
    log = []
    
    # 第一次称重: {1,2,3,4} vs {5,6,7,8}
    w1 = weigh([1,2,3,4], [5,6,7,8])
    log.append(f"W1: {{1,2,3,4}} vs {{5,6,7,8}} = {w1}")
    
    if w1 == 'B':  # 情况A: 次品在 9-12
        w2 = weigh([9,10,11], [1,2,3])
        log.append(f"W2: {{9,10,11}} vs {{1,2,3}} = {w2}")
        
        if w2 == 'B':  # 次品是 12
            w3 = weigh([12], [1])
            log.append(f"W3: {{12}} vs {{1}} = {w3}")
            return (12, w3 == 'L', log)
        elif w2 == 'L':  # 9,10,11中有偏重的
            w3 = weigh([9], [10])
            log.append(f"W3: {{9}} vs {{10}} = {w3}")
            if w3 == 'L': return (9, True, log)
            elif w3 == 'R': return (10, True, log)
            else: return (11, True, log)
        else:  # 9,10,11中有偏轻的
            w3 = weigh([9], [10])
            log.append(f"W3: {{9}} vs {{10}} = {w3}")
            if w3 == 'R': return (9, False, log)
            elif w3 == 'L': return (10, False, log)
            else: return (11, False, log)
    
    elif w1 == 'L':  # 情况B: 左重
        # 第二次: {1,2,5} vs {3,6,9}
        w2 = weigh([1,2,5], [3,6,9])
        log.append(f"W2: {{1,2,5}} vs {{3,6,9}} = {w2}")
        
        if w2 == 'B':  # 次品在 {4重, 7轻, 8轻}
            w3 = weigh([7], [8])
            log.append(f"W3: {{7}} vs {{8}} = {w3}")
            if w3 == 'R': return (7, False, log)
            elif w3 == 'L': return (8, False, log)
            else: return (4, True, log)
        elif w2 == 'L':  # 次品在 {1重, 2重, 6轻}
            w3 = weigh([1], [2])
            log.append(f"W3: {{1}} vs {{2}} = {w3}")
            if w3 == 'L': return (1, True, log)
            elif w3 == 'R': return (2, True, log)
            else: return (6, False, log)
        else:  # w2 == 'R': 次品在 {3重, 5轻}
            w3 = weigh([5], [9])
            log.append(f"W3: {{5}} vs {{9}} = {w3}")
            if w3 == 'R': return (5, False, log)
            else: return (3, True, log)
    
    else:  # w1 == 'R': 情况C (与B对称)
        w2 = weigh([5,6,1], [7,2,9])
        log.append(f"W2: {{5,6,1}} vs {{7,2,9}} = {w2}")
        
        if w2 == 'B':  # 次品在 {8重, 3轻, 4轻}
            w3 = weigh([3], [4])
            log.append(f"W3: {{3}} vs {{4}} = {w3}")
            if w3 == 'R': return (3, False, log)
            elif w3 == 'L': return (4, False, log)
            else: return (8, True, log)
        elif w2 == 'L':  # 次品在 {7重, 5偏重? no...}
            # W1: right heavy → 5,6,7,8 side heavy
            # Possible: 5H,6H,7H,8H or 1L,2L,3L,4L
            # W2: {5,6,1} vs {7,2,9} → left heavy
            # → 次品在 {5重,6重,2轻}
            w3 = weigh([5], [6])
            log.append(f"W3: {{5}} vs {{6}} = {w3}")
            if w3 == 'L': return (5, True, log)
            elif w3 == 'R': return (6, True, log)
            else: return (2, False, log)
        else:  # w2 == 'R': 次品在 {7重, 1轻}
            w3 = weigh([1], [9])
            log.append(f"W3: {{1}} vs {{9}} = {w3}")
            if w3 == 'R': return (1, False, log)
            else: return (7, True, log)

# 测试所有 24 种情况
print("📊 步骤 5: Python 暴力验证所有 24 种情况")
print("=" * 60)
all_correct = True
for ball in range(1, 13):
    for heavy in [True, False]:
        found_ball, found_heavy, log = solve_12_balls(ball, heavy)
        correct = (found_ball == ball) and (found_heavy == heavy)
        if not correct:
            all_correct = False
            print(f"  ❌ 球{ball}({'重' if heavy else '轻'}): "
                  f"找到 球{found_ball}({'重' if found_heavy else '轻'})")

if all_correct:
    print(f"  ✅ 所有 24 种情况全部正确识别!")
    print(f"  每种情况恰好使用 3 次称重")

# 展示几个例子
print(f"\n  示例:")
for ball, heavy in [(3, True), (7, False), (12, True)]:
    found_ball, found_heavy, log = solve_12_balls(ball, heavy)
    h_str = '偏重' if heavy else '偏轻'
    print(f"\n  次品: 球{ball} ({h_str})")
    for step in log:
        print(f"    {step}")
    print(f"    → 结论: 球{found_ball} {'偏重' if found_heavy else '偏轻'} ✅")

In [ ]:
# ========== 步骤 6: 可视化 - 决策树 ==========
fig, ax = plt.subplots(figsize=(16, 10))

# 画简化版决策树 (只展示情况A分支)
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')

# Root
ax.text(8, 9.5, 'W1: {1,2,3,4} vs {5,6,7,8}', ha='center', fontsize=11,
        fontweight='bold', bbox=dict(boxstyle='round,pad=0.4', facecolor='steelblue',
                                     alpha=0.8, edgecolor='black'),
        color='white')

# Level 1 branches
branches_l1 = [
    (2.5, 7.0, 'Balance\nDefect in 9-12', '#2ecc71'),
    (8.0, 7.0, 'Left Heavy\n1-4 heavy or 5-8 light', '#e67e22'),
    (13.5, 7.0, 'Right Heavy\n(symmetric)', '#e74c3c'),
]

for x, y, text, color in branches_l1:
    ax.annotate('', xy=(x, y+0.5), xytext=(8, 9.0),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    ax.text(x, y, text, ha='center', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.7),
            color='white')

# Level 2: Expand Balance branch
ax.text(2.5, 5.5, 'W2: {9,10,11} vs {1,2,3}', ha='center', fontsize=9,
        fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='steelblue',
                                     alpha=0.6), color='white')
ax.annotate('', xy=(2.5, 5.8), xytext=(2.5, 6.4),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# Level 2 branches
l2_branches = [
    (0.5, 4.0, 'Bal: #12\nW3: 12 vs 1', '#2ecc71'),
    (2.5, 4.0, 'Left: 9,10,11\nheavy\nW3: 9 vs 10', '#e67e22'),
    (4.5, 4.0, 'Right: 9,10,11\nlight\nW3: 9 vs 10', '#e74c3c'),
]

for x, y, text, color in l2_branches:
    ax.annotate('', xy=(x, y+0.6), xytext=(2.5, 5.2),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.2))
    ax.text(x, y, text, ha='center', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.5))

# Level 3: Final answers for balance→left branch
l3_answers = [
    (0.5, 2.5, '12H or 12L'),
    (1.7, 2.5, '9H'),
    (2.5, 2.5, '10H'),
    (3.3, 2.5, '11H'),
    (3.7, 2.5, '9L'),
    (4.5, 2.5, '10L'),
    (5.3, 2.5, '11L'),
]

for x, y, text in l3_answers:
    ax.text(x, y, text, ha='center', fontsize=7,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', edgecolor='gray'))

# Expand Left Heavy branch similarly
ax.text(8.0, 5.5, 'W2: {1,2,5} vs {3,6,9}', ha='center', fontsize=9,
        fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='steelblue',
                                     alpha=0.6), color='white')
ax.annotate('', xy=(8.0, 5.8), xytext=(8.0, 6.4),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

l2b_branches = [
    (6.5, 4.0, 'Bal: {4H,7L,8L}\nW3: 7 vs 8', '#2ecc71'),
    (8.0, 4.0, 'Left: {1H,2H,6L}\nW3: 1 vs 2', '#e67e22'),
    (9.5, 4.0, 'Right: {3H,5L}\nW3: 5 vs 9', '#e74c3c'),
]

for x, y, text, color in l2b_branches:
    ax.annotate('', xy=(x, y+0.6), xytext=(8.0, 5.2),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.2))
    ax.text(x, y, text, ha='center', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.5))

# Info box
textstr = ('Decision Tree for 12-Ball Problem\n'
           '24 possible cases, 3 weighings\n'
           'Each weighing splits into 3 branches\n'
           '3^3 = 27 >= 24 (sufficient)')
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(13.5, 4.5, textstr, fontsize=10, verticalalignment='top',
        bbox=props)

ax.set_title('12-Ball Defective Ball: Decision Tree (Partial)', 
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  决策树展示了3次称重的完整策略")
print(f"  每个内部节点是一次称重, 有3个分支 (左重/平衡/右重)")
print(f"  叶子节点是最终答案 (哪个球, 偏重还是偏轻)")
print(f"  树的深度 = 3 = 称重次数")

---

## 3. 谜题二：25 匹马赛跑 (Horse Racing)

### 🎯 场景设定

你有 **25 匹马**，每匹马的速度不同。赛道一次最多跑 **5 匹马**。

你**没有计时器**，只能看每场比赛中马的**相对排名**。

**问题**：最少需要几场比赛才能找出**最快的 3 匹马**？

### 📐 下界分析

- 每场比赛最多对 5 匹马排序 → 排除 2 匹（第 4、5 名确定不在前 3）
- 需要排除 22 匹马 → 至少 $\lceil 22/2 \rceil = 11$ 场？
- 但第一轮 5 场比赛可以初步分组，后续可以更高效

**答案是 7 场**。让我们一步步推导。

In [ ]:
# ========== 步骤 1: 第一轮 - 5 组各自比赛 (5场) ==========
print("📊 步骤 1: 第一轮 - 分组赛 (5 场)")
print("=" * 60)
print("  将 25 匹马分成 5 组, 每组 5 匹, 各自比赛:")
print()
print("  Race 1: A1 > A2 > A3 > A4 > A5")
print("  Race 2: B1 > B2 > B3 > B4 > B5")
print("  Race 3: C1 > C2 > C3 > C4 > C5")
print("  Race 4: D1 > D2 > D3 > D4 > D5")
print("  Race 5: E1 > E2 > E3 > E4 > E5")
print()
print("  (X1 = 组内第一, X2 = 组内第二, ...)")
print()
print("  💡 结论: 每组的第4、第5名不可能是全场前3")
print("     → 排除 10 匹马 (A4,A5,B4,B5,...,E4,E5)")
print("     → 剩余 15 匹候选")
print(f"     累计比赛: 5 场")

In [ ]:
# ========== 步骤 2: 第二轮 - 冠军赛 (第6场) ==========
print("📊 步骤 2: 第二轮 - 各组冠军赛 (第 6 场)")
print("=" * 60)
print("  Race 6: A1, B1, C1, D1, E1 比赛")
print("  假设结果: A1 > B1 > C1 > D1 > E1")
print()
print("  💡 关键推理:")
print("  ┌──────────────────────────────────────────────────┐")
print("  │ 1. A1 是全场最快的马 (确定!)                      │")
print("  │    - A1 比同组所有马快, 又比所有组冠军快           │")
print("  │                                                   │")
print("  │ 2. D 组和 E 组全部排除                            │")
print("  │    - D1 是第4, 则 D 组没有马能进前3               │")
print("  │    - E1 是第5, 则 E 组没有马能进前3               │")
print("  │    → 排除 D1,D2,D3 + E1,E2,E3 = 6匹             │")
print("  │                                                   │")
print("  │ 3. C 组只保留 C1                                  │")
print("  │    - C1 最好是全场第3, C2,C3 不可能进前3          │")
print("  │    → 排除 C2, C3                                  │")
print("  │                                                   │")
print("  │ 4. B 组保留 B1, B2                                │")
print("  │    - B1 最好是全场第2, B2 最好第3                  │")
print("  │    - B3 不可能: B1>B2>B3 且 A1>B1 → B3最好第4    │")
print("  │    → 排除 B3                                      │")
print("  │                                                   │")
print("  │ 5. A 组保留 A2, A3                                │")
print("  │    - A1 已确认第1, 但 A2,A3 仍可能是全场第2,3     │")
print("  └──────────────────────────────────────────────────┘")
print()
print("  剩余候选 (争夺第2、第3):")
print("    A2, A3, B1, B2, C1  → 恰好 5 匹!")
print(f"  累计比赛: 6 场")

In [ ]:
# ========== 步骤 3: 第三轮 - 最终决赛 (第7场) ==========
print("📊 步骤 3: 第三轮 - 决赛 (第 7 场)")
print("=" * 60)
print("  Race 7: A2, A3, B1, B2, C1 比赛")
print("  取前 2 名 → 加上已确认的 A1 → 得到全场前 3")
print()
print("  ✅ 总计: 7 场比赛!")
print()
print("  💡 排除逻辑的可视化:")
print("        A1  A2  A3  A4  A5")
print("        B1  B2  B3  B4  B5")
print("        C1  C2  C3  C4  C5")
print("        D1  D2  D3  D4  D5")
print("        E1  E2  E3  E4  E5")
print()
print("  第一轮后排除 (X4,X5):    [灰色]")
print("  第六场后排除 (D,E组 + B3 + C2,C3): [灰色]")
print("  已确认: A1 = 全场第1   [金色]")
print("  第七场候选: A2,A3,B1,B2,C1  [蓝色]")

In [ ]:
# ========== 步骤 4: 可视化 - 排除过程 ==========
fig, ax = plt.subplots(figsize=(12, 7))

# 5x5 grid of horses
groups = ['A', 'B', 'C', 'D', 'E']
# Status: 'eliminated', 'champion', 'finalist', 'normal'
status = {}
for g in groups:
    for rank in range(1, 6):
        name = f'{g}{rank}'
        if rank >= 4:  # Eliminated in round 1
            status[name] = 'elim_r1'
        else:
            status[name] = 'normal'

# After race 6
status['A1'] = 'champion'  # Confirmed #1
for name in ['D1','D2','D3','E1','E2','E3','C2','C3','B3']:
    status[name] = 'elim_r6'
for name in ['A2','A3','B1','B2','C1']:
    status[name] = 'finalist'

color_map = {
    'champion': '#FFD700',   # Gold
    'finalist': 'steelblue', # Blue
    'elim_r1': '#d3d3d3',    # Light gray
    'elim_r6': '#a9a9a9',    # Dark gray
    'normal': '#f0f0f0',     # Very light
}

for i, g in enumerate(groups):
    for j in range(5):
        name = f'{g}{j+1}'
        x, y = j, 4 - i
        color = color_map[status[name]]
        rect = plt.Rectangle((x - 0.4, y - 0.35), 0.8, 0.7,
                             facecolor=color, edgecolor='black', linewidth=1.5)
        ax.add_patch(rect)
        
        fontcolor = 'white' if status[name] in ['finalist', 'elim_r6'] else 'black'
        ax.text(x, y, name, ha='center', va='center', fontsize=11,
                fontweight='bold', color=fontcolor)

# Labels
ax.set_xlim(-0.8, 5)
ax.set_ylim(-0.8, 5.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('25 Horses: Elimination Process After 6 Races', 
             fontsize=14, fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#FFD700', edgecolor='black', label='Confirmed #1 (A1)'),
    Patch(facecolor='steelblue', edgecolor='black', label='Race 7 Finalists'),
    Patch(facecolor='#d3d3d3', edgecolor='black', label='Eliminated (Round 1: rank 4,5)'),
    Patch(facecolor='#a9a9a9', edgecolor='black', label='Eliminated (Race 6 logic)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

# Arrow showing race 6 ranking
ax.annotate('Race 6 ranking: A1 > B1 > C1 > D1 > E1',
           xy=(2, 5.2), ha='center', fontsize=11, fontweight='bold',
           color='steelblue')

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  金色: A1 已确认全场第1")
print(f"  蓝色: 5匹决赛马 (A2,A3,B1,B2,C1)")
print(f"  浅灰: 第一轮被排除 (组内第4,5名)")
print(f"  深灰: 第6场后被排除 (D组,E组全部 + B3 + C2,C3)")
print(f"  关键: 排除需要严格的逻辑推理, 不能遗漏任何候选!")

In [ ]:
# ========== 步骤 5: Python 模拟验证 7 场足够 ==========
def simulate_horse_racing(n_simulations=100000):
    """
    模拟 25 匹马赛跑, 验证 7 场策略总能找到前3
    """
    success = 0
    
    for _ in range(n_simulations):
        # 随机生成 25 匹马的速度 (越小越快)
        speeds = np.random.permutation(25)  # 0=最快, 24=最慢
        # 分成 5 组
        groups = speeds.reshape(5, 5)
        
        # Round 1: 各组内排序 (5 races)
        sorted_groups = np.sort(groups, axis=1)  # 每行排序
        group_order = np.argsort(groups, axis=1)  # 排名索引
        
        # Round 2: 各组冠军 (第0列) 比赛
        champions = sorted_groups[:, 0]  # 5个组冠军的速度
        champ_rank = np.argsort(champions)  # 冠军间的排名
        
        # 重新编号: 按冠军赛排名, A=最快组, ..., E=最慢组
        # A1 确定是全场第1
        best_group = champ_rank[0]
        
        # 决赛候选: A2, A3, B1, B2, C1
        A = sorted_groups[champ_rank[0]]  # A组 (冠军最快)
        B = sorted_groups[champ_rank[1]]  # B组
        C = sorted_groups[champ_rank[2]]  # C组
        
        finalists = [A[1], A[2], B[0], B[1], C[0]]
        
        # Race 7: 取决赛前2名
        finalists_sorted = sorted(finalists)
        top3_speeds = sorted([A[0], finalists_sorted[0], finalists_sorted[1]])
        
        # 验证: 这是否真的是全场前3?
        true_top3 = sorted(speeds)[:3]
        if top3_speeds == list(true_top3):
            success += 1
    
    return success / n_simulations

accuracy = simulate_horse_racing()

print("📊 步骤 5: Monte Carlo 模拟验证")
print("=" * 50)
print(f"  模拟次数: 100,000")
print(f"  每次随机分配 25 匹马的速度")
print(f"")
print(f"  7 场策略成功率: {accuracy:.4%}")
print(f"")
if accuracy > 0.9999:
    print(f"  ✅ 100% 成功! 7 场策略在所有情况下都能找到前 3")
else:
    print(f"  策略成功率: {accuracy:.4%}")

print(f"\n💡 这证明了 7 场是充分的")
print(f"   接下来验证 6 场是否不够...")

In [ ]:
# ========== 步骤 6: 证明 6 场不够 ==========
print("📊 步骤 6: 为什么 6 场不够?")
print("=" * 50)
print("  论证:")
print("  - 前5场: 必须让每匹马至少参加1场 (否则无法比较)")
print("  - 25匹马, 每场5匹 → 至少 5 场让所有马参赛")
print("  - 5场分组赛后: 只知道组内排名, 不知道组间关系")
print("  - 第6场: 最多比较5匹马")
print()
print("  5场分组赛后剩余候选 (前3名 × 5组) = 15 匹")
print("  第6场(冠军赛): 确定全场第1, 排除 D,E 组")
print("    剩余候选: A2,A3,B1,B2,C1 = 5 匹")
print("  但此时已用完 6 场, 无法再比赛!")
print()
print("  如果第6场不是冠军赛, 而是混合其他马?")
print("  → 无论怎么安排, 6场后总有 >3 匹马无法确定相对顺序")
print()
print("  ✅ 结论: 7 场是最少的!")
print()
print("  💡 严格证明需要信息论下界:")
print(f"    前5场: 获得5个组内排序 (但组间无信息)")
print(f"    第6场: 获得1个组间排序")
print(f"    剩余不确定性仍需至少1场 → 总计 7 场")

---

## 4. 谜题三：过河问题 (River Crossing)

### 🎯 场景设定

一个农夫要带着**狼 (Wolf)**、**鸡 (Chicken)** 和**米 (Grain)** 过河。

规则：
- 船一次只能载农夫和**一件物品**（或空船回来）
- 如果农夫不在场：
  - **狼会吃鸡** (Wolf eats Chicken)
  - **鸡会吃米** (Chicken eats Grain)
- 狼不吃米

**问题**：如何让所有三样安全过河？

### 💡 建模思路

将问题建模为**状态空间搜索**：
- **状态**：每个物体（农夫、狼、鸡、米）在左岸或右岸
- **动作**：农夫带一件物品（或不带）过河
- **约束**：任何时刻不能出现危险组合
- **目标**：从全在左岸 → 全在右岸

In [ ]:
# ========== 步骤 1: 状态空间建模 ==========
print("📊 步骤 1: 状态空间建模")
print("=" * 50)
print("  状态表示: (Farmer, Wolf, Chicken, Grain)")
print("    L = 左岸, R = 右岸")
print()
print("  初始状态: (L, L, L, L) — 全在左岸")
print("  目标状态: (R, R, R, R) — 全在右岸")
print()
print("  可能的状态数: 2^4 = 16")
print("  但很多状态是非法的 (危险组合)")
print()

# 枚举所有状态
all_states = list(product(['L', 'R'], repeat=4))
print(f"  所有 {len(all_states)} 种状态:")

def is_safe(state):
    """检查状态是否安全"""
    farmer, wolf, chicken, grain = state
    # 狼和鸡在同一边, 农夫不在 → 不安全
    if wolf == chicken and farmer != wolf:
        return False
    # 鸡和米在同一边, 农夫不在 → 不安全
    if chicken == grain and farmer != chicken:
        return False
    return True

safe_count = 0
unsafe_count = 0
for s in all_states:
    safe = is_safe(s)
    if safe:
        safe_count += 1
    else:
        unsafe_count += 1

print(f"    安全状态: {safe_count} 种")
print(f"    危险状态: {unsafe_count} 种")

In [ ]:
# ========== 步骤 2: BFS 搜索最短路径 ==========
def get_moves(state):
    """获取所有合法的下一步"""
    farmer, wolf, chicken, grain = state
    items = list(state)
    new_side = 'R' if farmer == 'L' else 'L'
    
    moves = []
    # 农夫自己过河 (不带东西)
    new_state = list(state)
    new_state[0] = new_side
    moves.append((tuple(new_state), 'Farmer alone'))
    
    # 农夫带一件物品
    names = ['Farmer', 'Wolf', 'Chicken', 'Grain']
    for i in range(1, 4):
        if items[i] == farmer:  # 物品在农夫同侧
            new_state = list(state)
            new_state[0] = new_side
            new_state[i] = new_side
            moves.append((tuple(new_state), f'Farmer + {names[i]}'))
    
    # 过滤掉不安全的状态
    safe_moves = [(s, m) for s, m in moves if is_safe(s)]
    return safe_moves

def bfs_river_crossing():
    """BFS 搜索最短过河方案"""
    start = ('L', 'L', 'L', 'L')
    goal = ('R', 'R', 'R', 'R')
    
    queue = deque([(start, [(start, 'Start')])])
    visited = {start}
    all_solutions = []
    min_steps = float('inf')
    
    while queue:
        current, path = queue.popleft()
        
        if len(path) - 1 > min_steps:
            continue
        
        if current == goal:
            if len(path) - 1 <= min_steps:
                min_steps = len(path) - 1
                all_solutions.append(path)
            continue
        
        for next_state, move in get_moves(current):
            if next_state not in visited:
                visited.add(next_state)
                queue.append((next_state, path + [(next_state, move)]))
    
    return all_solutions

solutions = bfs_river_crossing()

print("📊 步骤 2: BFS 搜索结果")
print("=" * 60)
print(f"  找到 {len(solutions)} 种最短方案")
print(f"  最少步骤: {len(solutions[0]) - 1}")
print()

# 展示第一种方案
print("  方案 1 (详细):")
print(f"  {'步骤':>4}  {'动作':<25}  {'左岸':<15}  {'右岸':<15}")
print(f"  {'─'*4}  {'─'*25}  {'─'*15}  {'─'*15}")

names_map = {0: 'F', 1: 'W', 2: 'C', 3: 'G'}
for step, (state, move) in enumerate(solutions[0]):
    left = [names_map[i] for i in range(4) if state[i] == 'L']
    right = [names_map[i] for i in range(4) if state[i] == 'R']
    left_str = ','.join(left) if left else '(empty)'
    right_str = ','.join(right) if right else '(empty)'
    print(f"  {step:>4}  {move:<25}  {left_str:<15}  {right_str:<15}")

print(f"\n  💡 F=Farmer, W=Wolf, C=Chicken, G=Grain")
print(f"     关键: 必须先送鸡过去 (因为鸡与狼、米都冲突)")

In [ ]:
# ========== 步骤 3: 可视化 - 状态转移图 ==========
fig, ax = plt.subplots(figsize=(14, 8))

# 手动布局最优路径的节点
path = solutions[0]
n_steps = len(path)

# 布局: 每步一列
x_positions = np.linspace(1, 13, n_steps)
y_base = 4

for step, (state, move) in enumerate(path):
    x = x_positions[step]
    
    # 画状态框
    left = [names_map[i] for i in range(4) if state[i] == 'L']
    right = [names_map[i] for i in range(4) if state[i] == 'R']
    
    # 左岸
    left_text = ','.join(left) if left else '-'
    right_text = ','.join(right) if right else '-'
    
    # 状态框
    state_text = f'L: {left_text}\n~river~\nR: {right_text}'
    color = '#2ecc71' if step == n_steps - 1 else 'steelblue' if step == 0 else '#e67e22'
    ax.text(x, y_base, state_text, ha='center', va='center', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.4', facecolor=color, alpha=0.7,
                     edgecolor='black'), color='white', fontweight='bold')
    
    # 动作标签
    if step > 0:
        direction = '→' if state[0] == 'R' else '←'
        ax.text((x_positions[step-1] + x) / 2, y_base + 1.3,
                f'{direction} {move}', ha='center', fontsize=8,
                color='black', fontstyle='italic')
        # 箭头
        ax.annotate('', xy=(x - 0.5, y_base), xytext=(x_positions[step-1] + 0.5, y_base),
                   arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    
    # 步骤编号
    ax.text(x, y_base - 1.5, f'Step {step}', ha='center', fontsize=9, fontweight='bold')

ax.set_xlim(0, 14)
ax.set_ylim(1, 7)
ax.axis('off')
ax.set_title('River Crossing: Optimal Solution (7 Steps)', fontsize=14, fontweight='bold')

# 安全规则
rules_text = ('Safety Rules:\n'
              'Wolf + Chicken (no Farmer) = Danger\n'
              'Chicken + Grain (no Farmer) = Danger')
ax.text(7, 6.5, rules_text, ha='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  蓝色: 初始状态 (全在左岸)")
print(f"  橙色: 中间状态")
print(f"  绿色: 目标状态 (全在右岸)")
print(f"  每一步标注了农夫的动作 (带谁过河/返回)")
print(f"  关键: 鸡必须在第一步被送走, 因为它与狼和米都冲突")

In [ ]:
# ========== 步骤 4: 状态空间完整图 ==========
print("📊 步骤 4: 完整状态空间分析")
print("=" * 50)

# 列出所有安全状态
safe_states = [s for s in all_states if is_safe(s)]
unsafe_states = [s for s in all_states if not is_safe(s)]

print(f"  安全状态 ({len(safe_states)} 种):")
for s in safe_states:
    left = [names_map[i] for i in range(4) if s[i] == 'L']
    right = [names_map[i] for i in range(4) if s[i] == 'R']
    print(f"    L={','.join(left) if left else '-':>8}  |  R={','.join(right) if right else '-':<8}")

print(f"\n  危险状态 ({len(unsafe_states)} 种):")
for s in unsafe_states:
    left = [names_map[i] for i in range(4) if s[i] == 'L']
    right = [names_map[i] for i in range(4) if s[i] == 'R']
    farmer_side = s[0]
    danger = []
    if s[1] == s[2] and s[0] != s[1]:
        danger.append('W eats C')
    if s[2] == s[3] and s[0] != s[2]:
        danger.append('C eats G')
    print(f"    L={','.join(left) if left else '-':>8}  |  R={','.join(right) if right else '-':<8}  ❌ {', '.join(danger)}")

print(f"\n💡 BFS 只在安全状态中搜索, 自动避开所有危险状态")

---

## 5. 方法论总结

### 📝 三个谜题的逻辑推理对比

| 维度 | 12球称重 | 25匹马赛跑 | 过河问题 |
|------|----------|------------|----------|
| 建模方式 | 决策树 | 锦标赛/排除法 | 状态空间图 |
| 搜索策略 | 三分法 (最大信息) | 分组→冠军赛→决赛 | BFS (最短路径) |
| 下界分析 | $3^3 \geq 24$ | 排除法计数 | 安全状态枚举 |
| 验证方法 | 穷举 24 种情况 | Monte Carlo 模拟 | BFS 自动搜索 |
| 核心洞察 | 交换球获取额外信息 | 冠军赛排除整组 | 鸡是关键冲突点 |

---

## 6. 核心概念回顾

### 📌 决策树 (Decision Tree)
- **定义**: 将多步决策过程表示为树结构，每个节点是一个决策，分支是可能的结果
- **应用**: 12球称重——每次称重是一个节点，3个分支对应3种结果
- **关键**: 树的深度 = 操作次数，叶子数 $\geq$ 可能情况数

### 📌 信息论下界 (Information-Theoretic Lower Bound)
- **定义**: 每次操作最多获取有限信息，总信息量决定了操作次数的下界
- **公式**: 最少操作次数 $\geq \lceil \log_k(\text{可能性数}) \rceil$，其中 $k$ = 每次结果数
- **应用**: $\lceil \log_3(24) \rceil = 3$ 次称重

### 📌 状态空间搜索 (State Space Search)
- **定义**: 将问题建模为状态转移图，用搜索算法找到从初始到目标的路径
- **BFS**: 找最短路径（最少步骤）
- **应用**: 过河问题——16个可能状态，10个安全状态，BFS找最短7步路径

### 📌 排除法 (Elimination)
- **定义**: 通过逻辑推理，系统地排除不可能的候选，缩小解空间
- **应用**: 25匹马——每场比赛排除一些马，最终7场找到前3
- **关键**: 明确每次操作能排除多少候选

### 🔗 完整流程
```
理解约束 → 建立模型 → 计算下界 → 设计策略 → 验证正确性
    ↓          ↓          ↓          ↓           ↓
  列出规则   决策树/图  信息论分析  最优分支    Python穷举
```

---

## 7. 常见误区

### ❌ 误区 1: 12球问题中，第一次称重应该 6 vs 6
**✓ 正确理解**: 6 vs 6 称重结果永远不会平衡（因为次品一定在某一边），浪费了"平衡"这个有价值的结果。正确做法是 4 vs 4，让不称的 4 个球利用"平衡"结果来缩小范围。每次称重应该把可能性分成**三等份**，而非两等份。

### ❌ 误区 2: 25匹马问题中，前5场用完就已经知道全场排名
**✓ 正确理解**: 前5场只给出了**组内排名**，不同组之间没有任何比较信息。A组第1可能比B组第3还慢——必须通过冠军赛建立组间联系，再通过逻辑排除缩小范围。

### ❌ 误区 3: 过河问题中，先送最危险的狼过河
**✓ 正确理解**: 应该先送**鸡**过河。因为鸡同时与狼和米冲突——如果鸡留在任何一边且农夫不在，就会出问题。狼和米之间没有冲突，可以安全地待在一起。

### ❌ 误区 4: 逻辑推理题不需要数学
**✓ 正确理解**: 虽然不需要高等数学，但信息论（下界分析）、图论（状态空间搜索）、组合数学（排列组合计数）都是强大的工具。它们帮助你证明"为什么N次操作足够"和"为什么N-1次不够"。

### ❌ 误区 5: 找到一种可行方案就够了
**✓ 正确理解**: 面试中通常要求**最优方案**（最少步骤/次数），并且要**证明最优性**。找到7步方案不够，还要论证为什么6步不可能。这需要下界分析或反例构造。